# Pol-ISARSpaceTarget: DINOv3 and ResNet18 Magnitude-Backbone Study

This notebook compares frozen visual backbones under one classification
protocol. It does not reuse weights trained on TFPSSTD. VV magnitude is read
directly from the complex `.dat` files. Scaling statistics are fitted on the
Pol-ISAR training split only and reused unchanged for validation and test.

Outputs: split audit, train-only statistics, feature matrices, validation
model selection, held-out metrics, confusion matrices, and DINOv3 patch-feature
saliency maps. DINOv3 is the current official Meta release; there is no
verified official DINOv4 checkpoint used here.



In [ ]:
!pip -q install -U "transformers>=4.56" scikit-learn seaborn



In [ ]:
import json, os, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from torchvision.models import ResNet18_Weights, resnet18

try:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive mount message:', exc)
except Exception:
    pass

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
ROOTS = [
    Path('/content/drive/MyDrive/drdo/Pol_ISARSpaceTarget1.0'),
    Path('/content/drive/MyDrive/drdo/Pol_ISARSpaceTarget1.0/Pol_ISARSpaceTarget1.0'),
    Path('/content/Pol_ISARSpaceTarget1.0'),
]
ROOT = next((p for p in ROOTS if (p/'ImageSets/images').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Dataset root containing ImageSets/images was not found.')
OUT = Path('/content/drive/MyDrive/drdo/polisar_backbone_study')
OUT.mkdir(parents=True, exist_ok=True)
print('dataset:', ROOT, 'output:', OUT)



## Duplicate-safe index and held-out protocol



In [ ]:
CLASSES = ['Capella','HISEA1','QPSSAR','Radarsat2','StarLink','Worldview4']
CLASS_TO_ID = {name:i for i,name in enumerate(CLASSES)}

def class_id(base):
    return next(i for name,i in CLASS_TO_ID.items() if base.startswith(name+'_'))

def index_split(split):
    folder = ROOT/'ImageSets/images'/split
    bases = sorted({p.name.rsplit('_',1)[0] for p in folder.glob('*_VV.dat')
                    if not re.search(r' \(\d+\)_VV\.dat$', p.name)})
    rows = [(folder, b, class_id(b)) for b in bases if (folder/f'{b}_VV.dat').exists()]
    expected = {'train':1314,'valid':162,'test':162}[split]
    assert len(rows) == expected, f'{split}: expected {expected}, found {len(rows)}'
    return rows

splits = {s:index_split(s) for s in ['train','valid','test']}
display(pd.DataFrame([{'split':s,'images':len(v)} for s,v in splits.items()]))

def read_vv(path):
    raw = np.fromfile(path, np.float32)
    assert raw.size == 256*256*2, (path, raw.size)
    return raw[0::2].reshape(256,256) + 1j*raw[1::2].reshape(256,256)



## Fit normalization on training data only



In [ ]:
def fit_logmag_stats(rows):
    total = total2 = 0.0
    count = 0
    for i,(folder,base,_) in enumerate(rows,1):
        x = np.log1p(np.abs(read_vv(folder/f'{base}_VV.dat'))).astype(np.float64)
        total += x.sum(); total2 += np.square(x).sum(); count += x.size
        if i % 200 == 0: print('stats', i, '/', len(rows))
    mean = total/count
    std = max((total2/count - mean*mean)**0.5, 1e-6)
    return float(mean), float(std)

stats_path = OUT/'vv_logmag_train_stats.json'
if stats_path.exists():
    stats = json.loads(stats_path.read_text())
    MU, SIGMA = stats['mean'], stats['std']
else:
    MU, SIGMA = fit_logmag_stats(splits['train'])
    stats_path.write_text(json.dumps({'mean':MU,'std':SIGMA,'source':'train only'}, indent=2))
print('train-only mean/std:', MU, SIGMA)

IMAGENET_MEAN = torch.tensor([.485,.456,.406]).view(3,1,1)
IMAGENET_STD = torch.tensor([.229,.224,.225]).view(3,1,1)

def radar_tensor(folder, base, size=224):
    x = np.log1p(np.abs(read_vv(folder/f'{base}_VV.dat'))).astype(np.float32)
    x = np.clip((x-MU)/SIGMA, -3, 3)
    x = torch.from_numpy((x+3)/6).unsqueeze(0).repeat(3,1,1)
    x = F.interpolate(x.unsqueeze(0), (size,size), mode='bilinear', align_corners=False).squeeze(0)
    return (x-IMAGENET_MEAN)/IMAGENET_STD



## Frozen backbones

DINOv3 weights are license-gated. Accept the model license on Hugging Face,
create a read token, and place it in Colab Secrets as `HF_TOKEN`. The notebook
never prints the token. ResNet18 remains runnable if DINO access is pending.



In [ ]:
from transformers import AutoModel

class ResNetFeatures(torch.nn.Module):
    def __init__(self):
        super().__init__()
        net = resnet18(weights=ResNet18_Weights.DEFAULT)
        net.fc = torch.nn.Identity()
        self.net = net
    def forward(self,x): return self.net(x)

def load_dinov3():
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except Exception:
            token = None
    if not token:
        raise RuntimeError('Set HF_TOKEN in Colab Secrets after accepting the DINOv3 license.')
    return AutoModel.from_pretrained('facebook/dinov3-vits16-pretrain-lvd1689m', token=token)

backbones = {'ResNet18': ResNetFeatures().to(DEVICE).eval()}
try:
    backbones['DINOv3-ViT-S/16'] = load_dinov3().to(DEVICE).eval()
except Exception as exc:
    print('DINOv3 skipped:', exc)
for model in backbones.values():
    for p in model.parameters(): p.requires_grad_(False)

@torch.inference_mode()
def encode(model_name, model, rows, split, batch_size=32):
    cache = OUT/f"{model_name.replace('/','_')}_{split}_features.npz"
    if cache.exists():
        z=np.load(cache); return z['X'], z['y'], z['base']
    feats=[]; ys=[]; bases=[]
    for start in range(0,len(rows),batch_size):
        batch=rows[start:start+batch_size]
        x=torch.stack([radar_tensor(f,b) for f,b,_ in batch]).to(DEVICE)
        out=model(x)
        if model_name.startswith('DINO'):
            feat=out.last_hidden_state[:,0]
        else:
            feat=out
        feats.append(feat.float().cpu().numpy()); ys.extend(v for _,_,v in batch); bases.extend(b for _,b,_ in batch)
        print(model_name, split, min(start+len(batch),len(rows)), '/', len(rows))
    X=np.concatenate(feats); y=np.asarray(ys); base=np.asarray(bases)
    np.savez_compressed(cache,X=X,y=y,base=base)
    return X,y,base



## Linear-probe comparison and confusion matrices



In [ ]:
results=[]
for name,model in backbones.items():
    Xtr,ytr,_=encode(name,model,splits['train'],'train')
    Xva,yva,_=encode(name,model,splits['valid'],'valid')
    Xte,yte,_=encode(name,model,splits['test'],'test')
    best=None
    for C in [0.01,0.1,1.0,10.0]:
        clf=LogisticRegression(C=C,max_iter=3000,class_weight='balanced',random_state=42).fit(Xtr,ytr)
        va=accuracy_score(yva,clf.predict(Xva))
        if best is None or va>best[0]: best=(va,C,clf)
    _,C,clf=best
    pred=clf.predict(Xte)
    p,r,f,_=precision_recall_fscore_support(yte,pred,average='macro',zero_division=0)
    row={'backbone':name,'C':C,'accuracy':accuracy_score(yte,pred),'precision_macro':p,'recall_macro':r,'f1_macro':f}
    results.append(row)
    cm=confusion_matrix(yte,pred)
    pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(OUT/f"{name.replace('/','_')}_confusion_matrix.csv")
    plt.figure(figsize=(7,6)); sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=CLASSES,yticklabels=CLASSES)
    plt.title(f'{name} held-out test confusion matrix'); plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout()
    plt.savefig(OUT/f"{name.replace('/','_')}_confusion_matrix.png",dpi=180); plt.show()
results_df=pd.DataFrame(results).sort_values('f1_macro',ascending=False)
display(results_df); results_df.to_csv(OUT/'backbone_comparison.csv',index=False)



## DINOv3 patch-feature saliency on one held-out image

This is the spatial norm of patch embeddings, not a fabricated attention
matrix. It shows where the frozen backbone has strong local feature response.



In [ ]:
if 'DINOv3-ViT-S/16' in backbones:
    folder,base,label=splits['test'][0]
    x=radar_tensor(folder,base).unsqueeze(0).to(DEVICE)
    with torch.inference_mode(): tokens=backbones['DINOv3-ViT-S/16'](x).last_hidden_state
    patch=tokens[:,5:] if tokens.shape[1]==201 else tokens[:,1:]
    side=int(round(patch.shape[1]**0.5)); sal=patch.norm(dim=-1).reshape(1,1,side,side)
    sal=F.interpolate(sal,(256,256),mode='bilinear',align_corners=False).squeeze().cpu().numpy()
    raw=np.log1p(np.abs(read_vv(folder/f'{base}_VV.dat')))
    plt.figure(figsize=(11,4)); plt.subplot(1,3,1); plt.imshow(raw,cmap='gray'); plt.title(base); plt.axis('off')
    plt.subplot(1,3,2); plt.imshow(sal,cmap='inferno'); plt.title('Patch-feature saliency'); plt.axis('off')
    plt.subplot(1,3,3); plt.imshow(raw,cmap='gray'); plt.imshow(sal,cmap='jet',alpha=.45); plt.title('Overlay'); plt.axis('off')
    plt.tight_layout(); plt.savefig(OUT/'dinov3_patch_feature_saliency.png',dpi=200,bbox_inches='tight'); plt.show()



## Interpretation rule

Report DINOv3 only if the gated checkpoint was loaded and its result row was
produced. Compare it with ResNet18 under this identical frozen-feature protocol.
Do not compare these linear-probe numbers directly with the earlier end-to-end
ComplexCNN result without stating that the training protocols differ.
